In [1]:
import jax
import jax.numpy as jnp
from function_utils import construct_func_from_intensity_matrix
from function_utils import construct_func_from_payment_matrix
from probability_callbacks import ProbabilityCallbacks
from prototype_1 import semimarkov_solver as pt1_mks
from prototype_1_heun import semimarkov_solver as pt1_mks_heun
from prototype_2 import semimarkov_solver as pt2_mks
from prototype_3 import semimarkov_solver as pt3_mks
from prototype_4 import semimarkov_solver as pt4_mks
from prototype_5 import semimarkov_solver as pt5_mks
from functools import partial

#jax.config.update("jax_enable_x64", True)

In [2]:
@jax.jit
def mu_3_scale(t, u, scale):
    my_poly2 = jnp.polyval(
        jnp.array([
            -3.677247e+03,
            0,
            9.898868e+04,
            -4.121887e+05,
            8.566131e+05,
            -1.111126e+06,
            9.761716e+05,
            -6.033659e+05,
            2.671941e+05,
            -8.548107e+04,
            1.987502e+04,
            -3.406968e+03,
            4.481199e+02,
            -4.880314e+01,
            4.753792
        ]),
        u / 30
    ) * 0.01
    
    my_poly = jnp.polyval(
        jnp.array([
            -3.677247e+03,
            0,
            9.898868e+04,
            -4.121887e+05,
            8.566131e+05,
            -1.111126e+06,
            9.761716e+05,
            -6.033659e+05,
            2.671941e+05,
            -8.548107e+04,
            1.987502e+04,
            -3.406968e+03,
            4.481199e+02,
            -4.880314e+01,
            4.753792
        ]),
        u / 30
    ) * 0.09
    
    return scale * (my_poly + my_poly2)

@jax.jit
def mu_3(t, u):
    my_poly2 = jnp.polyval(
        jnp.array([
            -3.677247e+03,
            0,
            9.898868e+04,
            -4.121887e+05,
            8.566131e+05,
            -1.111126e+06,
            9.761716e+05,
            -6.033659e+05,
            2.671941e+05,
            -8.548107e+04,
            1.987502e+04,
            -3.406968e+03,
            4.481199e+02,
            -4.880314e+01,
            4.753792
        ]),
        u / 30
    ) * 0.01
    
    my_poly = jnp.polyval(
        jnp.array([
            -3.677247e+03,
            0,
            9.898868e+04,
            -4.121887e+05,
            8.566131e+05,
            -1.111126e+06,
            9.761716e+05,
            -6.033659e+05,
            2.671941e+05,
            -8.548107e+04,
            1.987502e+04,
            -3.406968e+03,
            4.481199e+02,
            -4.880314e+01,
            4.753792
        ]),
        u / 30
    ) * 0.09
    
    return my_poly + my_poly2

#@jax.jit
#def mu_3(t, u):
#    return 0.15 + 0 * u

@jax.jit
def unit_payment(t, u):
    return 0 * t + 0 * u * 0 + 1

In [3]:
intensity_matrix = [
    [None, mu_3, mu_3, mu_3, None],
    [None, None, mu_3, mu_3, mu_3],
    [None, None, None, None, None],
    [None, None, None, None, None],
    [None, None, None, None, None]
]

intensity_matrix_scale = [
    [None, mu_3_scale, mu_3_scale, mu_3_scale, None],
    [None, None, mu_3_scale, mu_3_scale, mu_3_scale],
    [None, None, None, None, None],
    [None, None, None, None, None],
    [None, None, None, None, None]
]
"""
intensity_matrix = [
    [None, mu_3],
    [None, None]
]
"""

intensity = construct_func_from_intensity_matrix(intensity_matrix)
intensity_scale = construct_func_from_intensity_matrix(intensity_matrix_scale)
step_size = 1 / 24

u = jnp.linspace(0, 30, 30 * 12 + 1, endpoint=True)

u = jnp.expand_dims(u, axis=0)
scale = jnp.expand_dims(jnp.linspace(0.95, 1.05, 100), 1)
intensity_kwargs = {'scale': scale}

In [19]:
intensity_matrix_p = (
    (None, mu_3, mu_3, mu_3, None),
    (None, None, mu_3, mu_3, mu_3),
    (None, None, None, None, None),
    (None, None, None, None, None),
    (None, None, None, None, None)
)

intensity_matrix_p_scale = (
    (None, mu_3_scale, mu_3_scale, mu_3_scale, None),
    (None, None, mu_3_scale, mu_3_scale, mu_3_scale),
    (None, None, None, None, None),
    (None, None, None, None, None),
    (None, None, None, None, None)
)

In [20]:
# Baseline with Euler scheme
def bench1():
    result = pt1_mks(grid=u,
                intensity=intensity,
                prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

# Improved difference version (euler scheme)
def bench2():
    result = pt2_mks(units=30,
                 discretization_unit=12,
                 intensity=intensity,
                 prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

# With einsum
def bench3():
    result = pt3_mks(units=30,
                 discretization_unit=12,
                 intensity=intensity,
                 prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

# With Heun scheme
def bench4():
    result = pt4_mks(units=30,
                 discretization_unit=12,
                 intensity=intensity,
                 prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

# Baseline with heun scheme
def bench5():
    result = pt1_mks_heun(grid=u,
                intensity=intensity,
                prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

def bench6():
    result = pt5_mks(units=30,
                 discretization_unit=12,
                 intensity=intensity_matrix_p,
                 prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

bench1()
bench2()
bench3()
bench4()
bench5()
bench6()
1+1

2

In [ ]:
%timeit -r 10 -n 100 bench1() # Baseline Euler
%timeit -r 10 -n 100 bench2() # New Euler
%timeit -r 10 -n 100 bench3() # New with einsum Euler
%timeit -r 10 -n 100 bench4() # New with einsum Heun
%timeit -r 10 -n 100 bench5() # Baseline Heun
%timeit -r 10 -n 100 bench6() # New with sparse loop Heun

7.16 ms ± 117 μs per loop (mean ± std. dev. of 10 runs, 100 loops each)
4.39 ms ± 7.84 μs per loop (mean ± std. dev. of 10 runs, 100 loops each)
4.39 ms ± 14.1 μs per loop (mean ± std. dev. of 10 runs, 100 loops each)
7.68 ms ± 5.39 μs per loop (mean ± std. dev. of 10 runs, 100 loops each)
14.4 ms ± 55.2 μs per loop (mean ± std. dev. of 10 runs, 100 loops each)
5.92 ms ± 5.32 μs per loop (mean ± std. dev. of 10 runs, 100 loops each)


In [22]:
def bench1_2():
    result = pt1_mks(grid=u,
                intensity=intensity_scale,
                intensity_kwargs=intensity_kwargs,
                prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

def bench2_2():
    result = pt2_mks(units=30,
                 discretization_unit=12,
                 intensity=intensity_scale,
                 intensity_kwargs=intensity_kwargs,
                 prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

def bench3_2():
    result = pt3_mks(units=30,
                 discretization_unit=12,
                 intensity=intensity_scale,
                 intensity_kwargs=intensity_kwargs,
                 prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

def bench4_2():
    result = pt4_mks(units=30,
                 discretization_unit=12,
                 intensity=intensity_scale,
                 intensity_kwargs=intensity_kwargs,
                 prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

def bench5_2():
    result = pt1_mks_heun(grid=u,
                intensity=intensity_scale,
                intensity_kwargs=intensity_kwargs,
                prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

def bench6_2():
    result = pt5_mks(units=30,
                 discretization_unit=12,
                 intensity=intensity_matrix_p_scale,
                 intensity_kwargs=intensity_kwargs,
                 prob_callback='default')

    p, p0 = result['probability']
    
    return p.block_until_ready()

bench1_2()
bench2_2()
bench3_2()
bench4_2()
bench5_2()
bench6_2()
2+2

4

In [23]:
%timeit -r 10 -n 100 bench1_2() # Baseline 
%timeit -r 10 -n 100 bench2_2() # New Euler
%timeit -r 10 -n 100 bench3_2() # New with einsum Euler
%timeit -r 10 -n 100 bench4_2() # New with einsum Heun
%timeit -r 10 -n 100 bench5_2() # Baseline heun
%timeit -r 10 -n 100 bench6_2() # New with sparse loop Heun

16.1 ms ± 75.5 μs per loop (mean ± std. dev. of 10 runs, 100 loops each)
11.4 ms ± 80.7 μs per loop (mean ± std. dev. of 10 runs, 100 loops each)
11.5 ms ± 21.2 μs per loop (mean ± std. dev. of 10 runs, 100 loops each)
17.6 ms ± 57 μs per loop (mean ± std. dev. of 10 runs, 100 loops each)
29.1 ms ± 40.4 μs per loop (mean ± std. dev. of 10 runs, 100 loops each)
12.3 ms ± 16 μs per loop (mean ± std. dev. of 10 runs, 100 loops each)
